In [1]:
import pandas as pd
import re
import html
from tqdm import tqdm

tqdm.pandas()
print("Library berhasil dimuat")

Library berhasil dimuat


In [2]:
df = pd.read_csv('rawData/dataNews.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 234 entries, 0 to 233
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         234 non-null    object
 1   publish_date  234 non-null    object
 2   content       234 non-null    object
dtypes: object(3)
memory usage: 5.6+ KB


In [3]:
initial_len = len(df)
df.drop_duplicates(subset=['content'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Total berita: {len(df)} (dihapus {initial_len - len(df)} duplikat)")

Total berita: 234 (dihapus 0 duplikat)


In [4]:
def clean_news_bertopic(text):
    if not isinstance(text, str):
        return ""

    text = html.unescape(text)

    non_content_patterns = [
        r'(?i)^baca juga[:\-\s].*$',
        r'(?i)^simak juga[:\-\s].*$',
        r'(?i)^lihat juga[:\-\s].*$',
        r'(?i)^advertisement.*$',
        r'(?i)^scroll to continue.*$',
        r'(?i)^video[:\-\s].*$',
        r'(?i)^foto[:\-\s].*$',
        r'(?i)^reporter[:\-\s].*$',
        r'(?i)^editor[:\-\s].*$',
        r'(?i)^penulis[:\-\s].*$',
        r'(?i)^sumber[:\-\s].*$',
    ]
    for pattern in non_content_patterns:
        text = re.sub(pattern, '', text, flags=re.MULTILINE)

    text = re.sub(
        r'(?i)SCROLL TO CONTINUE WITH CONTENT|ADVERTISEMENT'
        r'|BACA JUGA:|SIMAK JUGA:|LIHAT JUGA:|VIDEO:|FOTO:'
        r'|REPORTER:|EDITOR:|PENULIS:|SUMBER:',
        '', text
    )

    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[""]', '"', text)
    text = re.sub(r"['']", "'", text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

print("Fungsi cleaning siap")

Fungsi cleaning siap


In [5]:
df['data_cleaning'] = df['content'].progress_apply(clean_news_bertopic)
df['case_folding']  = df['data_cleaning'].apply(lambda x: x.lower() if isinstance(x, str) else "")

print("Cleaning dan case folding selesai")

100%|██████████| 234/234 [00:00<00:00, 1314.24it/s]


Cleaning dan case folding selesai


In [6]:
normalization_dict = {
    'yg': 'yang', 'dg': 'dengan', 'dgn': 'dengan',
    'tsb': 'tersebut', 'tdk': 'tidak', 'tak': 'tidak',
    'krn': 'karena', 'sdh': 'sudah', 'blm': 'belum',
    'utk': 'untuk', 'thd': 'terhadap', 'ttg': 'tentang',
    'jd': 'jadi', 'pd': 'pada', 'dlm': 'dalam',
    'dpt': 'dapat', 'hrs': 'harus', 'msh': 'masih',
    'jk': 'jika', 'dr': 'dari', 'kpd': 'kepada',
    'spt': 'seperti', 'stlh': 'setelah', 'sblm': 'sebelum',
    'org': 'orang', 'pem': 'pemerintah',
    'pemda': 'pemerintah daerah',
    'pemkot': 'pemerintah kota',
    'pemkab': 'pemerintah kabupaten',
}

def normalize_minimal_news(text):
    if not isinstance(text, str):
        return ""
    words = text.split()
    normalized = []
    for word in words:
        replacement = normalization_dict.get(word)
        if replacement is None:
            normalized.append(word)
        elif replacement:
            normalized.append(replacement)
    return ' '.join(normalized)

df['normalization'] = df['case_folding'].progress_apply(normalize_minimal_news)

print(f"Normalisasi selesai — {len(normalization_dict)} istilah kamus")

100%|██████████| 234/234 [00:00<00:00, 14626.50it/s]

Normalisasi selesai — 29 istilah kamus


In [7]:
min_len = 30
initial = len(df)
df = df[df['normalization'].str.len() >= min_len]
df.reset_index(drop=True, inplace=True)

print(f"Dokumen dihapus : {initial - len(df)} (panjang < {min_len} karakter)")
print(f"Dokumen tersisa : {len(df)}")

Dokumen dihapus : 0 (panjang < 30 karakter)
Dokumen tersisa : 234


In [8]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING — DATA BERITA (BERTopic)")
print("=" * 60)

df['orig_length']  = df['content'].str.len()
df['final_length'] = df['normalization'].str.len()
df['orig_words']   = df['content'].apply(lambda x: len(str(x).split()))
df['final_words']  = df['normalization'].apply(lambda x: len(str(x).split()))

print(f"\nStatistik panjang teks:")
print(f"  Rata-rata panjang asli  : {df['orig_length'].mean():.1f} karakter")
print(f"  Rata-rata panjang akhir : {df['final_length'].mean():.1f} karakter")
print(f"  Reduksi                 : {((df['orig_length'].mean() - df['final_length'].mean()) / df['orig_length'].mean() * 100):.1f}%")

print(f"\nStatistik jumlah kata:")
print(f"  Rata-rata kata asli  : {df['orig_words'].mean():.1f}")
print(f"  Rata-rata kata akhir : {df['final_words'].mean():.1f}")
print(f"  Reduksi              : {((df['orig_words'].mean() - df['final_words'].mean()) / df['orig_words'].mean() * 100):.1f}%")

print(f"\nDistribusi panjang teks akhir:")
print(f"  Terpendek : {df['final_length'].min()} karakter")
print(f"  Terpanjang: {df['final_length'].max()} karakter")
print(f"  Median    : {df['final_length'].median():.1f} karakter")

for i in range(min(2, len(df))):
    print(f"\nBERITA {i+1}:")
    print("[ORIGINAL]")
    print(df['content'].iloc[i][:300] + "...")
    print("[NORMALIZATION — hasil akhir]")
    print(df['normalization'].iloc[i][:300] + "...")
    print("-" * 60)

LAPORAN KUALITAS PREPROCESSING — DATA BERITA (BERTopic)

Statistik panjang teks:
  Rata-rata panjang asli  : 3042.1 karakter
  Rata-rata panjang akhir : 2850.5 karakter
  Reduksi                 : 6.3%

Statistik jumlah kata:
  Rata-rata kata asli  : 416.6
  Rata-rata kata akhir : 404.4
  Reduksi              : 2.9%

Distribusi panjang teks akhir:
  Terpendek : 254 karakter
  Terpanjang: 25371 karakter
  Median    : 2309.5 karakter

BERITA 1:
[ORIGINAL]
Korps Pemberantasan Tindak Pidana Korupsi (Kortas Tipikor) Polri mengungkap kasus dugaan korupsi pada Ditjen Energi Baru Terbarukan dan Konservasi Energi (EBTKE) Kementerian ESDM terkait pengadaan penerangan jalan umum tenaga surya (PJUTS). Polri menyebut kerugian dalam kasus ini Rp 19.522.256.578,7...
[NORMALIZATION — hasil akhir]
korps pemberantasan tindak pidana korupsi kortas tipikor polri mengungkap kasus dugaan korupsi pada ditjen energi baru terbarukan dan konservasi energi ebtke kementerian esdm terkait pengadaan penerangan jala

In [9]:
bertopic_data = df[[
    'content', 'data_cleaning', 'case_folding', 'normalization'
]].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['content'])} -> final:{len(row['normalization'])}",
    axis=1
)

output_file = "bertopic_pisah_data/news_bertopic_without_stop_word.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data diekspor ke   : {output_file}")
print(f"Jumlah dokumen siap: {len(bertopic_data)}")
print(f"Kolom              : {bertopic_data.columns.tolist()}")

Data diekspor ke   : bertopic_pisah_data/news_bertopic_without_stop_word.csv
Jumlah dokumen siap: 234
Kolom              : ['content', 'data_cleaning', 'case_folding', 'normalization', 'preprocessing_stats']
